# Bank Marketing Campaign ROI Analysis
### Who to Call, When to Call, and Why It Matters

**Author:** [Your Name]  
**Date:** March 2026

**Objective:** Analyze a Portuguese bank's direct marketing campaign data to identify which customers are most likely to subscribe to a term deposit, optimize campaign strategy, and quantify the ROI impact of targeted marketing.

**Dataset:** Bank Marketing Dataset (45,211 records) — phone-based campaign data with customer demographics, campaign details, and subscription outcomes.

---
## Part 1: Setup & Data Loading

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve,
    classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
try:
    plt.style.use('seaborn-v0_8')
except OSError:
    plt.style.use('seaborn')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# Load the dataset
# Note: This CSV uses semicolons (;) as separators, not commas
df = pd.read_csv('data/raw/bank-full.csv', sep=';')

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Dataset shape: 45211 rows x 17 columns
Memory usage: 25.75 MB


In [3]:
# First look at the data
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [4]:
# Last 5 rows
df.tail()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,unknown,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,unknown,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,unknown,no
45210,37,entrepreneur,married,secondary,no,2971,no,no,cellular,17,nov,361,2,188,11,other,no


In [5]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


In [6]:
# Statistical summary for numeric columns
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [7]:
# Statistical summary for categorical columns
df.describe(include='object')

,job,marital,education,default,housing,loan,contact,month,poutcome,y
count,45211,45211,45211,45211,45211,45211,45211,45211,45211,45211
unique,12,3,4,2,2,2,3,12,4,2
top,blue-collar,married,secondary,no,yes,no,cellular,may,unknown,no
freq,9732,27214,23202,44396,25130,37967,29285,13766,36959,39922


In [8]:
# Check column names and unique values count
print("Column names and unique value counts:")
print("-" * 40)
for col in df.columns:
    print(f"{col:15s} | {df[col].nunique():5d} unique | dtype: {df[col].dtype}")

Column names and unique value counts:
----------------------------------------
age             |    77 unique | dtype: int64
job             |    12 unique | dtype: object
marital         |     3 unique | dtype: object
education       |     4 unique | dtype: object
default         |     2 unique | dtype: object
balance         |  7168 unique | dtype: int64
housing         |     2 unique | dtype: object
loan            |     2 unique | dtype: object
contact         |     3 unique | dtype: object
day             |    31 unique | dtype: int64
month           |    12 unique | dtype: object
duration        |  1573 unique | dtype: int64
campaign        |    48 unique | dtype: int64
pdays           |   559 unique | dtype: int64
previous        |    41 unique | dtype: int64
poutcome        |     4 unique | dtype: object
y               |     2 unique | dtype: object


---
## Part 2: Data Cleaning
Let's check for missing values, duplicates, and data quality issues.

In [9]:
# Check for missing values
print("Missing values per column:")
print("-" * 40)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found!")
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

Missing values per column:
----------------------------------------
No missing values found!

Total missing values: 0


In [10]:
# Check for 'unknown' values which act as missing data in this dataset
print("'unknown' values per column:")
print("-" * 40)
for col in df.select_dtypes(include='object').columns:
    unknown_count = (df[col] == 'unknown').sum()
    if unknown_count > 0:
        pct = unknown_count / len(df) * 100
        print(f"{col:15s} | {unknown_count:5d} unknowns ({pct:.1f}%)")

'unknown' values per column:
----------------------------------------
job             |   288 unknowns (0.6%)
education       |  1857 unknowns (4.1%)
contact         | 13020 unknowns (28.8%)
poutcome        | 36959 unknowns (81.7%)


In [11]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")
if duplicates > 0:
    print("Removing duplicates...")
    df = df.drop_duplicates()
    print(f"New shape: {df.shape}")

Duplicate rows: 0


In [12]:
# Handle 'unknown' values
# Strategy: Keep them — they represent real-world data gaps
# We'll treat 'unknown' as its own category rather than dropping rows
# This is more honest and preserves sample size

# Convert 'pdays' sentinel value: -1 means "never contacted before"
# Create a more useful binary flag
df['previously_contacted'] = (df['pdays'] != -1).astype(int)
print(f"Previously contacted: {df['previously_contacted'].sum()} ({df['previously_contacted'].mean()*100:.1f}%)")
print(f"Never contacted: {(df['previously_contacted'] == 0).sum()} ({(1 - df['previously_contacted'].mean())*100:.1f}%)")

Previously contacted: 8257 (18.3%)
Never contacted: 36954 (81.7%)


In [13]:
# Encode target variable: 'yes' -> 1, 'no' -> 0
df['y_numeric'] = (df['y'] == 'yes').astype(int)
print(f"Subscription rate: {df['y_numeric'].mean()*100:.1f}%")
print(f"  Yes: {df['y_numeric'].sum()}")
print(f"  No:  {(df['y_numeric'] == 0).sum()}")

Subscription rate: 11.7%
  Yes: 5289
  No:  39922


In [14]:
# Create age groups for later segmentation
df['age_group'] = pd.cut(df['age'],
                         bins=[17, 30, 45, 60, 100],
                         labels=['18-30', '31-45', '46-60', '60+'])
print("Age group distribution:")
print(df['age_group'].value_counts().sort_index())

Age group distribution:
age_group
18-30     7030
31-45    23733
46-60    13260
60+       1188
Name: count, dtype: int64


In [15]:
# Create balance quartiles for segmentation
df['balance_group'] = pd.qcut(df['balance'], q=4,
                               labels=['Low', 'Medium', 'High', 'Very High'],
                               duplicates='drop')
print("Balance group distribution:")
print(df['balance_group'].value_counts())

Balance group distribution:
balance_group
Low          11317
High         11306
Very High    11297
Medium       11291
Name: count, dtype: int64


In [16]:
# Save cleaned dataset
df.to_csv('data/cleaned/bank-cleaned.csv', index=False)
print(f"Cleaned dataset saved: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nNew columns added: age_group, balance_group, previously_contacted, y_numeric")

Cleaned dataset saved: 45211 rows x 21 columns

New columns added: age_group, balance_group, previously_contacted, y_numeric
